# Self-Supervised Learning: Exercises

Test your understanding of self-supervised learning principles, InfoNCE loss, and paradigm tradeoffs.

In [1]:
import random

import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

from ml_first_principles.ssl_models import InfoNCELoss

## Exercise 1: Hand-Computing InfoNCE

Given a batch size $N=2$ (so $2N=4$ augmented views total). Let the embeddings for the views be $z_1, z_2$ for sample A and $z_3, z_4$ for sample B. 
Suppose the cosine similarity matrix $S$ (shape 4x4) is:

$$S = \begin{bmatrix} 
1.0 & 0.8 & -0.1 & -0.2 \\ 
0.8 & 1.0 & -0.2 & 0.1 \\ 
-0.1 & -0.2 & 1.0 & 0.9 \\ 
-0.2 & 0.1 & 0.9 & 1.0 
\end{bmatrix}$$

For $\tau = 0.5$, compute the InfoNCE loss for the anchor $z_1$. Ignore self-similarity.

**Hint:** Use the formula $\ell = -\log \frac{\exp(sim(z_1, z_2)/\tau)}{\exp(sim(z_1, z_2)/\tau) + \exp(sim(z_1, z_3)/\tau) + \exp(sim(z_1, z_4)/\tau)}$.

In [2]:
# Exercise 1 inputs — derive the anchor-z1 loss by hand before opening the solution below.
S = np.array([
    [1.0, 0.8, -0.1, -0.2],
    [0.8, 1.0, -0.2, 0.1],
    [-0.1, -0.2, 1.0, 0.9],
    [-0.2, 0.1, 0.9, 1.0],
])
tau = 0.5

### Solution

For anchor $z_1$ the positive is $z_2$ and the negatives are $z_3, z_4$ (self-similarity $S_{11}$ is excluded).

1. Scale the relevant similarities by $\tau = 0.5$ (division rule):
   $S_{12}/\tau = 0.8/0.5 = 1.6$, $\quad S_{13}/\tau = -0.1/0.5 = -0.2$, $\quad S_{14}/\tau = -0.2/0.5 = -0.4$.
2. Exponentiate each term: $e^{1.6} \approx 4.9530$, $\quad e^{-0.2} \approx 0.8187$, $\quad e^{-0.4} \approx 0.6703$.
3. Softmax probability of the positive (sum the denominator first):
   $p = \dfrac{4.9530}{4.9530 + 0.8187 + 0.6703} = \dfrac{4.9530}{6.4421} \approx 0.7689$.
4. Negative log-likelihood: $\ell_1 = -\log p \approx -\log(0.7689) \approx 0.2629$.

**Result:** $\ell_1 = -\log \dfrac{e^{1.6}}{e^{1.6} + e^{-0.2} + e^{-0.4}} \approx 0.2629$

In [3]:
# Deterministic verification of the hand derivation
exp_pos = np.exp(S[0, 1] / tau)
exp_negs = np.exp(S[0, 2] / tau) + np.exp(S[0, 3] / tau)
loss_z1 = -np.log(exp_pos / (exp_pos + exp_negs))

print(f"Computed loss for anchor z1: {loss_z1:.4f}")
assert np.isclose(loss_z1, 0.2629, atol=1e-4), "Hand-derived value does not match"

Computed loss for anchor z1: 0.2629


## Exercise 2: Implementing NT-Xent Symmetry

The InfoNCE loss in SimCLR is computed symmetrically for both views (i.e., using view 1 as anchor, then view 2 as anchor). 

Implement a function `symmetric_nt_xent` that takes the similarity matrix directly and computes the total average loss over all $2N$ anchors. Verify that the result matches expectations.

In [4]:
def symmetric_nt_xent(sim_matrix, tau=1.0):
    """
    sim_matrix: (2N, 2N) pairwise cosine similarity matrix
    tau: temperature
    Assumes the block layout: the first N rows are view 1, the next N rows
    are view 2, so the positive partner of anchor i is (i + N) mod 2N.
    """
    _2N = sim_matrix.shape[0]
    N = _2N // 2

    # Scale by tau
    sim = sim_matrix / tau

    # Mask self-similarity
    np.fill_diagonal(sim, -np.inf)

    # Construct labels (positive pairs)
    labels = np.zeros(_2N, dtype=int)
    labels[:N] = np.arange(N, _2N)
    labels[N:] = np.arange(N)

    # Softmax
    exp_sim = np.exp(sim)
    probs = exp_sim / np.sum(exp_sim, axis=1, keepdims=True)

    # Gather positive probabilities
    pos_probs = probs[np.arange(_2N), labels]

    return -np.mean(np.log(pos_probs))

# Exercise 1 lists the embeddings as (z1, z2, z3, z4) = (A-view1, A-view2, B-view1, B-view2),
# while symmetric_nt_xent expects the block layout (A-view1, B-view1 | A-view2, B-view2).
# Permute rows and columns of S into that layout with the order (z1, z3, z2, z4).
order = np.array([0, 2, 1, 3])
S_blocks = S[np.ix_(order, order)]

loss = symmetric_nt_xent(S_blocks, tau=tau)
print(f"Symmetric NT-Xent loss: {loss:.4f}")

# Deterministic check: the mean of the four per-anchor losses (anchor z1 alone gave 0.2629)
assert np.isclose(loss, 0.2696, atol=1e-4), "Symmetric loss does not match the hand value"

# Cross-check against the unit-tested package reference on seeded embeddings,
# where the setup matches: the package takes the two views directly and computes
# the same symmetric NT-Xent (L2-normalizing internally).
v1 = rng.standard_normal((6, 3))
v2 = rng.standard_normal((6, 3))
v1_unit = v1 / np.linalg.norm(v1, axis=1, keepdims=True)
v2_unit = v2 / np.linalg.norm(v2, axis=1, keepdims=True)
Z = np.vstack([v1_unit, v2_unit])

loss_matrix = symmetric_nt_xent(Z @ Z.T, tau=0.5)
loss_ref = InfoNCELoss(temperature=0.5).forward(v1, v2)
print(f"symmetric_nt_xent: {loss_matrix:.6f} | package InfoNCELoss: {loss_ref:.6f}")
assert np.isclose(loss_matrix, loss_ref, atol=1e-8), "Does not match the package reference"

Symmetric NT-Xent loss: 0.2696
symmetric_nt_xent: 2.987693 | package InfoNCELoss: 2.987693


## Exercise 3: Conceptual Analysis of SSL Paradigms

**Question:**
Compare Contrastive Learning (e.g., SimCLR), Non-Contrastive Learning (e.g., BYOL), and Masked Image Modeling (e.g., MAE) across the following dimensions:
1. Dependence on Data Augmentation
2. Need for Negative Samples
3. Semantic Level of Representations

**Answer:**
1. **Dependence on Data Augmentation:**
   - SimCLR/BYOL: Highly dependent. They rely on augmentations to define semantic invariance. Poor augmentations (e.g., missing color jitter) lead to degenerate representations.
   - MAE: Low dependence. Masking serves as the primary data corruption; complex structural augmentations are less critical.
2. **Need for Negative Samples:**
   - SimCLR: Yes. Requires massive batch sizes or momentum queues to provide sufficient negatives to prevent collapse.
   - BYOL/MAE: No. BYOL uses asymmetric architectures (EMA) to prevent collapse, while MAE prevents collapse via the generative pixel-reconstruction objective.
3. **Semantic Level of Representations:**
   - SimCLR/BYOL: Captures highly abstracted, global semantic features (great for classification) but often loses fine-grained spatial information.
   - MAE: Captures dense, localized features useful for pixel-level tasks (e.g., segmentation) as well as global structures, bridging the gap between localized reconstruction and global understanding.